# 00 Infrastructure and port check (stage G0)

The Gwangyang study (2012-12-20 to 2026-10-06) runs the Imsil stages with the **current Haenam rules** (the Haenam renewal tree, moho `27.Haenam_sequence_renewal`):
- one sensor per station for the whole window;
- one dated orientation table;
- S differential times on T after rotation;
- template horizontals turned into each scanned day's orientation;
- local magnitudes without station terms.

The code was copied from the renewal tree, not from the Imsil port, because the renewal already holds these rules and Imsil's grid alignment.

This notebook checks three things:
1. **Port check.** Every copied file either equals its Haenam source, or carries `GWANGYANG PORT` markers at every change.
2. **Registry.** The Gwangyang registry resolves every product, and the 11 KMA events are in UTC.
3. **Environments.** moho's base, fmf and pq-gpu environments have the packages the stages need.

It ends with the decision log.

In [1]:
import os, sys, hashlib, subprocess
import pandas as pd
ROOT = '/home/msseo/works/73.Gwangyang_sequence'
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 120); pd.set_option('display.max_rows', 200)
sha = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
prov = pd.read_csv(f'{ROOT}/PROVENANCE.tsv', sep='\t', header=None, names=['path', 'source', 'sha256_copied'])
rows = []
for r in prov.itertuples():
    p = f'{ROOT}/{r.path}'
    if not os.path.exists(p):
        rows.append(dict(path=r.path, status='removed or renamed', markers=0)); continue
    now = sha(p); src_now = sha(r.source) if os.path.exists(r.source) else ''
    txt = open(p, errors='ignore').read()
    n = txt.count('GWANGYANG PORT')
    status = 'identical to Haenam' if now == r.sha256_copied else ('edited, marked' if n else 'EDITED WITHOUT MARKER')
    rows.append(dict(path=r.path, status=status, markers=n, haenam_source_changed_since_copy=bool(src_now and src_now != r.sha256_copied)))
pc = pd.DataFrame(rows)
display(pc.status.value_counts().to_frame('files'))
display(pc[pc.status != 'identical to Haenam'].sort_values('path'))
new = sorted(set(os.path.relpath(os.path.join(dp, f), ROOT) for dp, _, fs in os.walk(ROOT) for f in fs
                 if f.endswith(('.py', '.sh', '.csv')) and not dp.startswith(f'{ROOT}/data') and '/continuous' not in dp
                 and '__pycache__' not in dp and 'logs' not in dp and 'phasenet_plus' not in dp and 'notebooks' not in dp and 'relocation/hypodd' not in dp)
          - set(prov.path))
print('files added by this project (not copies):'); print('\n'.join('  ' + x for x in new))
PORT_OK = not (pc.status == 'EDITED WITHOUT MARKER').any()
print('PORT CHECK:', 'PASS' if PORT_OK else 'FAIL')

,files
status,
identical to Haenam,95
"edited, marked",12


,path,status,markers,haenam_source_changed_since_copy
0,detection/haenam_catalog.py,"edited, marked",5,False
7,detection/pipeline/config.py,"edited, marked",5,False
18,detection/run_renewal_detection.sh,"edited, marked",4,False
21,detection/station_table/build_stations_renewal.py,"edited, marked",4,False
30,detection/tools/link_continuous.py,"edited, marked",1,False
33,detection/tools/mirror_mara_streams.py,"edited, marked",7,False
35,detection/tools/r1c_orientation.py,"edited, marked",6,False
60,registry/haenam.py,"edited, marked",1,True
61,registry/orientation.py,"edited, marked",2,False
68,relocation/code/haenam_dl_cluster_config.py,"edited, marked",6,False


files added by this project (not copies):
  detection/station_table/add_link_until.py
  detection/station_table/stations.csv
  docs/decisions.csv
  docs/station_inventory/build_station_inventory.py
  docs/station_inventory/tables/orientation_periods.csv
  docs/station_inventory/tables/response_epochs.csv
  docs/station_inventory/tables/stations.csv
  docs/station_inventory/tables/stream_periods.csv
  registry/events.csv
  registry/gwangyang.py
  registry/haenam_haenam_original.py
  registry/layout.csv
  registry/streams.csv
PORT CHECK: PASS


In [2]:
sys.path.insert(0, f'{ROOT}/registry')
import gwangyang as R
reg = pd.DataFrame([dict(product=p, path=R.path(None, p), exists=os.path.exists(R.path(None, p))) for p in R.products()])
display(reg)
ev = pd.read_csv(f'{ROOT}/registry/events.csv', dtype={'eid': str, 'kma_id': str})
display(ev)
REG_OK = len(ev) == 11 and (pd.to_datetime(ev.origin_time_utc) == pd.to_datetime(ev.kma_id, format='%Y%m%d%H%M%S') - pd.Timedelta(hours=9)).all()
print('REGISTRY:', 'PASS' if REG_OK else 'FAIL', f'({len(ev)} events, UTC = KST - 9 h)')

,product,path,exists
0,dl_continuous,/home/msseo/works/73.Gwangyang_sequence/data/continuous/sds_links,False
1,dl_station_table,/home/msseo/works/73.Gwangyang_sequence/detection/station_table/stations.csv,True
2,dl_catalog,/home/msseo/works/73.Gwangyang_sequence/catalogs/Gwangyang_DL_catalog_2012-2026.csv,False
3,orientation_periods,/home/msseo/works/73.Gwangyang_sequence/docs/station_inventory/tables/orientation_periods.csv,True
4,kma_national_catalog,/home/msseo/works/73.Gwangyang_sequence/data/catalogs/kma_entire_catalog_260309.csv,True
5,reference_25m_dir,/home/msseo/works/73.Gwangyang_sequence/catalogs/reference_25m,False
6,tm_templates_195,/home/msseo/works/73.Gwangyang_sequence/catalogs/templates/Gwangyang_templates_ref25.csv,False
7,kma_necis_domestic,/home/msseo/works/73.Gwangyang_sequence/data/catalogs/necis_events_domestic_1978_20261013.csv,True


,key,eid,kma_id,origin_time_utc,latitude,longitude,ML,sequence
0,20210720T1614,20210720161418,20210721011418,2021-07-20 16:14:18,35.00,127.72,1.3,2021
1,20210827T0023,20210827002315,20210827092315,2021-08-27 00:23:15,35.00,127.72,1.6,2021
2,20210827T2203,20210827220322,20210828070322,2021-08-27 22:03:22,35.00,127.73,2.2,2021
3,20210828T1334,20210828133410,20210828223410,2021-08-28 13:34:10,35.01,127.73,1.7,2021
4,20210828T1339,20210828133922,20210828223922,2021-08-28 13:39:22,34.99,127.71,1.1,2021
5,20210828T2345,20210828234556,20210829084556,2021-08-28 23:45:56,35.00,127.72,1.5,2021
6,20260101T0249,20260101024947,20260101114947,2026-01-01 02:49:47,35.01,127.73,1.4,2026
7,20260101T0848,20260101084800,20260101174800,2026-01-01 08:48:00,35.00,127.72,1.4,2026
8,20260202T1941,20260202194119,20260203044119,2026-02-02 19:41:19,35.00,127.72,1.6,2026
9,20260203T0215,20260203021535,20260203111535,2026-02-03 02:15:35,35.00,127.72,1.1,2026


REGISTRY: PASS (11 events, UTC = KST - 9 h)


In [3]:
envs = {'base': '/home/msseo/miniforge3/bin/python', 'fmf': '/home/msseo/miniforge3/envs/fmf/bin/python', 'pq-gpu': '/home/msseo/miniforge3/envs/pq-gpu/bin/python'}
probe = {'base': 'import obspy, pyocto, torch, pandas; print(obspy.__version__, pyocto.__version__, torch.__version__, torch.cuda.is_available(), pandas.__version__)',
         'fmf': 'import fast_matched_filter as f, obspy; print(getattr(f, "__version__", "?"), obspy.__version__)',
         'pq-gpu': 'import torch, obspy; print(torch.__version__, torch.cuda.is_available(), obspy.__version__)'}
erows = []
for k, py in envs.items():
    r = subprocess.run([py, '-c', probe[k]], capture_output=True, text=True)
    erows.append(dict(env=k, python=py, versions=r.stdout.strip() or r.stderr.strip()[-200:], ok=r.returncode == 0))
envt = pd.DataFrame(erows); display(envt)
w = '/home/msseo/works/14.EQNet/EQNet/docs/model_phasenet_plus/model_99.pth'
WEIGHTS = sha(w); print('PhaseNet+ weights sha256:', WEIGHTS, '(the Haenam frozen weights f1643330...)' if WEIGHTS.startswith('f1643330') else '(DIFFERENT from Haenam)')
ENV_OK = bool(envt.ok.all()) and WEIGHTS.startswith('f1643330')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu); print('ENVIRONMENTS:', 'PASS' if ENV_OK else 'FAIL')

,env,python,versions,ok
0,base,/home/msseo/miniforge3/bin/python,1.4.2 0.1.10 2.12.0+cu130 True 2.2.2,True
1,fmf,/home/msseo/miniforge3/envs/fmf/bin/python,1.5.0 1.5.0,True
2,pq-gpu,/home/msseo/miniforge3/envs/pq-gpu/bin/python,2.11.0+cu128 True 1.4.2,True


PhaseNet+ weights sha256: f1643330e34bffaa4f7e3a5f7dd9205fb4019d76e5b9b6cb598232360d22f384 (the Haenam frozen weights f1643330...)
GPU: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition, 97887 MiB
ENVIRONMENTS: PASS


In [4]:
dec = pd.read_csv(f'{ROOT}/docs/decisions.csv')
display(dec[['id', 'setting', 'gwangyang_value', 'status']])

,id,setting,gwangyang_value,status
0,P1,study window,"2012-12-20 (first KS.GWYB day) .. 2026-10-06 (last day on MARA, 2026-10-15 listing)",decided
1,P2,machine and project folder,moho /home/msseo/works/73.Gwangyang_sequence,decided
2,P3,method rules,"the same: one sensor per station, one dated orientation table, S dt.cc on T, template N/E turned to each scanned day...",decided
3,P4,networks,KS + KG where data exist; JU not used,decided
4,R1,association box,"34.30-35.70 N, 126.93-128.53 E",rule
5,R2,source centroid,35.0000 N 127.7286 E,rule
6,R3,relocation selection,"within 5 km of 35.0000 N 127.7286 E, depth >= 10 km, same quality gate (relocation_selection.py)",rule
7,R4,station set,"116 KS/KG codes on MARA within 100 km with data in the window (108 KS, 8 KG)",rule
8,R5,known events,11 KMA events within 5 km in 2021 and 2026 (UTC),rule
9,G-D1,sensor per station,"the sensor rule for every multi-sensor station; current rule where an orientation was never measured (9 stations, re...",default


## Comprehensive summary

In [5]:
summ = pd.DataFrame([
    ('Port check (identical or marked edits)', 'PASS' if PORT_OK else 'FAIL', f"{(pc.status == 'identical to Haenam').sum()} identical, {(pc.status == 'edited, marked').sum()} edited with markers, {(pc.status == 'EDITED WITHOUT MARKER').sum()} unmarked"),
    ('Registry and events', 'PASS' if REG_OK else 'FAIL', f'{len(reg)} products; {len(ev)} KMA events (UTC)'),
    ('Environments and PhaseNet+ weights', 'PASS' if ENV_OK else 'FAIL', '; '.join(f'{r.env}: {r.versions}' for r in envt.itertuples())),
    ('Decision log', 'info', f"{len(dec)} rows: " + ', '.join(f'{k} {v}' for k, v in dec.status.value_counts().items())),
], columns=['check', 'result', 'detail'])
display(summ)
print('Take-homes:')
print('1. The Gwangyang code is the Haenam renewal code; every change is marked GWANGYANG PORT and listed above (region values, paths, the inventory and link rules).')
print('2. The registry serves Gwangyang paths through the same interface; the copied code imports it unchanged through the haenam.py shim.')
print('3. Detection uses the frozen Haenam PhaseNet+ weights; every stage runs in moho\'s existing environments.')

,check,result,detail
0,Port check (identical or marked edits),PASS,"95 identical, 12 edited with markers, 0 unmarked"
1,Registry and events,PASS,8 products; 11 KMA events (UTC)
2,Environments and PhaseNet+ weights,PASS,base: 1.4.2 0.1.10 2.12.0+cu130 True 2.2.2; fmf: 1.5.0 1.5.0; pq-gpu: 2.11.0+cu128 True 1.4.2
3,Decision log,info,"19 rows: default 8, rule 6, decided 4, open 1"


Take-homes:
1. The Gwangyang code is the Haenam renewal code; every change is marked GWANGYANG PORT and listed above (region values, paths, the inventory and link rules).
2. The registry serves Gwangyang paths through the same interface; the copied code imports it unchanged through the haenam.py shim.
3. Detection uses the frozen Haenam PhaseNet+ weights; every stage runs in moho's existing environments.
